#  Veteriner Asistanı — RAG Tabanlı Soru-Cevap

**Amaç:** Çiftçinin doğal dille soru sorabileceği (ör. "buzağım neden yem yemiyor?")
ve veteriner bilgi tabanından cevap alabileceği bir asistan kurmak.

**Yöntem — RAG (Retrieval-Augmented Generation):**
1. Bilgi tabanı: veteriner bilgi metinleri
2. Getirme: soruya en uygun metni bulma (embedding + benzerlik)
3. Cevap: bulunan metne dayanarak sade bir yanıt üretme

**Not:** Başlangıç için küçük bir örnek bilgi tabanıyla iskeleti kuruyoruz; sonra
gerçek veteriner dökümanlarıyla genişletilebilir.

In [1]:
!pip install sentence-transformers -q

`sentence-transformers` bu metinleri embeddinge (anlamı temsil eden sayı vektörüne) çevirmeye yarayan RAG'ın kalbi olan kısım burası.

In [2]:
# Veteriner bilgi tabanı (başlangıç — sonra genişletilebilir)
bilgi_tabani = [
    "Buzağıların yem yememesinin yaygın sebepleri arasında geçiş dönemi stresi, "
    "sindirim bozuklukları ve diş çıkarma dönemi bulunur. İştahsızlık iki günden "
    "uzun sürerse veteriner hekime başvurulmalıdır.",

    "Süt ineklerinde süt veriminin düşmesi; yetersiz beslenme, su tüketiminin "
    "azalması, sıcaklık stresi veya meme iltihabı (mastitis) kaynaklı olabilir. "
    "Yem kalitesi ve su erişimi ilk kontrol edilmesi gereken noktalardır.",

    "İneklerde topallık genellikle tırnak hastalıkları, zeminin sert veya ıslak "
    "olması ya da eklem iltihabından kaynaklanır. Erken fark edilirse tedavi "
    "başarısı yüksektir.",

    "Gebe ineklerin doğuma yakın döneminde beslenmesi kritiktir. Enerji ve protein "
    "ihtiyacı artar; ancak aşırı besleme doğum güçlüğüne yol açabilir. Doğum "
    "tarihinden 3 hafta önce geçiş rasyonuna başlanmalıdır.",

    "Hayvanlarda aşı takvimine uyulması hastalıkların önlenmesinde temeldir. Şap, "
    "brusella ve şarbon gibi hastalıklar için düzenli aşılama yapılmalı; aşı "
    "sonrası hayvan gözlem altında tutulmalıdır.",

    "Süt sağımı sonrası meme temizliği ve hijyen, mastitis riskini azaltır. Sağım "
    "ekipmanları düzenli temizlenmeli, ilk sütte pıhtı veya kan görülürse veteriner "
    "hekime danışılmalıdır."
]

print("Bilgi tabanındaki metin sayısı:", len(bilgi_tabani))
for i, m in enumerate(bilgi_tabani):
    print(f"{i+1}. {m[:50]}...")

Bilgi tabanındaki metin sayısı: 6
1. Buzağıların yem yememesinin yaygın sebepleri arası...
2. Süt ineklerinde süt veriminin düşmesi; yetersiz be...
3. İneklerde topallık genellikle tırnak hastalıkları,...
4. Gebe ineklerin doğuma yakın döneminde beslenmesi k...
5. Hayvanlarda aşı takvimine uyulması hastalıkların ö...
6. Süt sağımı sonrası meme temizliği ve hijyen, masti...


`bilgi_tabani `— asistanın "bildiği" her şey burada. Altı kısa veteriner metni: buzağı iştahsızlığı, süt verimi düşüşü, topallık, gebelik beslenmesi, aşı, meme sağlığı. Çiftçi bir soru sorduğunda, asistan bu metinler arasından en uygununu bulup ona göre cevap verecek.Bilgi tabanı büyüdükçe asistan akıllanır. Gerçek projede bunu yüzlerce veteriner dökümanıyla doldurup daha da akıllı ve iyi eğitilmiş bir asistan olmasını sağlayabiliriz..



In [3]:
from sentence_transformers import SentenceTransformer, util

# Embedding modelini yükle (metinleri anlam vektörüne çevirir)
embed_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# Bilgi tabanındaki tüm metinleri embedding'e çevir
bilgi_embed = embed_model.encode(bilgi_tabani, convert_to_tensor=True)

print("Bilgi tabanı embedding'e çevrildi.")
print("Her metin", bilgi_embed.shape[1], "boyutlu bir vektöre dönüştü.")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Bilgi tabanı embedding'e çevrildi.
Her metin 384 boyutlu bir vektöre dönüştü.


`SentenceTransformer("paraphrase-multilingual-MiniLM-...")` bu bir embedding modelidir.Multilingual çoklu dil demek onu seçtik.

`embed_model.encode(bilgi_tabani)` Bilgi modelindeki her metni bir sayı vektörüne dönüstürüyor.PC anlam karsılastıramaz ama sayıları karsılastırabilir ,iki sayı vektörü karsılastırılıp birbirine benziyorsa o zaman anlmaları da birbirine benziyo olabilir sonucuna varırız.


`KRAL-ERKEK+KADIN = KRALİÇE` mantığı vardır,Bu yüzde her veterinerlik metni artıkbir sayı vektörüne dönüşmüş durumda.



In [4]:
# Çiftçinin sorusu
soru = "buzağım neden yem yemiyor?"

# Soruyu da embedding'e çevir
soru_embed = embed_model.encode(soru, convert_to_tensor=True)

# Soruyu bilgi tabanındaki her metinle karşılaştır (kosinüs benzerliği)
benzerlikler = util.cos_sim(soru_embed, bilgi_embed)[0]

# En yakın metni bul
en_yakin_index = benzerlikler.argmax().item()
en_yakin_metin = bilgi_tabani[en_yakin_index]

print("SORU:", soru)
print()
print("EN UYGUN BİLGİ (otomatik bulundu):")
print(en_yakin_metin)
print()
print("Benzerlik skoru:", round(benzerlikler[en_yakin_index].item(), 3))

SORU: buzağım neden yem yemiyor?

EN UYGUN BİLGİ (otomatik bulundu):
Buzağıların yem yememesinin yaygın sebepleri arasında geçiş dönemi stresi, sindirim bozuklukları ve diş çıkarma dönemi bulunur. İştahsızlık iki günden uzun sürerse veteriner hekime başvurulmalıdır.

Benzerlik skoru: 0.414


`soru_embed = embed_model.encode(soru)` — çiftçinin sorusunu da aynı şekilde vektöre çeviriyoruz. Artık hem soru hem de 6 metin, aynı "anlam uzayında" sayı olarak duruyor.

`util.cos_sim(...)` —  Kosinüs benzerliği ile soruyu bilgi tabanındaki her metinle tek tek karşılaştırıyor: "bu soru hangi metne anlamca en yakın?" . Her metin için bir benzerlik skoru çıkıyor (1'e yakın = çok benzer).

`benzerlikler.argmax() `— en yüksek skorlu metni seçiyor. Yani "buzağım yem yemiyor" sorusuna, sistemin kendisi bilgi tabanından buzağı iştahsızlığı metnini bulmalı

asistanı tamamlıyoruz. Şu ana kadar sistem doğru bilgiyi buluyor ama sadece o metni olduğu gibi gösteriyor. Asıl asistan, bu bilgiyi alıp çiftçiye sade, sohbet gibi bir cevap yazmalı. Bunun için bir dil modeli (LLM) ekleyeceğiz.



In [5]:
from transformers import pipeline
import torch

# Küçük, hızlı bir Türkçe-uyumlu dil modeli yükle
llm = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Dil modeli yüklendi.")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Dil modeli yüklendi.


Qwen2.5-1.5B-Instruct modelini yüklüyoruz.Nispeten küçükmüş ama başalangıç seviyesi olarak iyi bir başlangıç olabilirmiş diye düşünüyorum.

`float16 ve device_map="auto"`da modeli GPU'da verimli çalıştırmak için.

In [6]:
def veteriner_asistan(soru):
    soru_embed = embed_model.encode(soru, convert_to_tensor=True)
    benzerlikler = util.cos_sim(soru_embed, bilgi_embed)[0]
    en_yakin = bilgi_tabani[benzerlikler.argmax().item()]

    mesaj = [
        {"role": "system", "content":
         "Sen bir veteriner asistanısın. SADECE sana verilen bilgiye dayanarak, "
         "çiftçiye sade ve anlaşılır bir dille cevap ver. Bilgi yoksa 'bu konuda "
         "yeterli bilgim yok, veteriner hekime danışın' de. Teşhis koyma."},
        {"role": "user", "content":
         f"Bilgi: {en_yakin}\n\nÇiftçinin sorusu: {soru}"}
    ]

    cevap = llm(mesaj, max_new_tokens=150, do_sample=False)
    return cevap[0]["generated_text"][-1]["content"]


soru = "buzağım neden yem yemiyor?"
print("SORU:", soru)
print()
print("ASİSTAN:", veteriner_asistan(soru))

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


SORU: buzağım neden yem yemiyor?



[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


ASİSTAN: Buzağı yem emin değil mi? Çiftçi, buzağı yem yememediğiniz için birkaç önemli neden vardır:

1. **Geçiş Dönemi Stresi**: Buzağı geçiş dönemine karşı durmadan yemek yapmanızı bekler. Bu durumda, buzağı yemekle zorlaşırlar ve buzağı yemeye devam etmeye kararlı olabilirler.

2. **Sindirim Bozukluğu**: Sindirim bozukluğu, buzağı yememe isteyenler için bir işaret olarak değerlendirilebilir. Bu da buzağı yemeye yönelik davranışlar oluşturabilir


In [7]:
sorular = [
    "ineğimin sütü azaldı ne yapmalıyım?",
    "hayvanıma ne zaman aşı yaptırmalıyım?",
    "ineğim topallıyor sebebi ne olabilir?"
]

for s in sorular:
    print("SORU:", s)
    print("ASİSTAN:", veteriner_asistan(s))
    print("-" * 60)

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


SORU: ineğimin sütü azaldı ne yapmalıyım?


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ASİSTAN: İneklerin sütünü azalttığı durumda, en doğru yolunuz, süt veriminizi artırmanızdır. Bu, ineklerin sağlıklı ve iyileşen bir şekilde yaşayabilmesini sağlar. İşte bazı öneriler:

1. İneklerinizin süt verisini artırırken, süt içeriğini artırmak için:
   - İneklerinizin süt verisinin yüksek olduğu yerlerde daha fazla süt içecek.
   - İneklerinizin süt verisinin düşük olduğu yerlerde daha az süt içecek.
   - İneklerinizin süt verisinin yüksek olduğu yerlerde
------------------------------------------------------------
SORU: hayvanıma ne zaman aşı yaptırmalıyım?


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ASİSTAN: Hayvanınız için aşı yapmanız gereken zaman belirtilmemiş. Ancak, aşı takvimini uygulamak, hastalıkların önlenmesi için önemli olabilir. Eğer hayvanınız şu anda hastalıklı durumda veya aşıya ihtiyaç duyuyorsa, veterinere başvurun. Ayrıca, aşı sonrası hayvanı gözlem altında tutmak önemlidir.
------------------------------------------------------------
SORU: ineğim topallıyor sebebi ne olabilir?
ASİSTAN: İneklerin topallaması, süt ineklerinde süt veriminin düşmesiyle ilgili olabilir. Bu durumda, yeterli beslenme, su tüketimi, sıcaklık stresi veya mastitis gibi hastalıklarla ilişkilidir. İneklerin yem kalitesi ve su erişimi ilk kontrol edilmesi gereken nokta olacaktır.
------------------------------------------------------------


Soru 2 (aşı): "Hayvanınız için aşı yapmanız gereken zaman belirtilmemiş. Ancak aşı takvimini uygulayın..." — bu iyi. Doğru bilgiyi (aşı metnini) bulmuş, mantıklı cevap vermiş. ✓

Soru 3 (topallık): "İneklerin topallaması, süt ineklerinde süt veriminin düşmesiyle ilgili olabilir..." — bu biraz karışık . Topallık metnini bulması gerekirken, cevabı süt verimine kaydırmış. Yani doğru konuya yakın ama tam oturmamış.

Soru 1 (süt azaldı): "İneklerin sütünü artırırken süt içeriğini artırmak için... süt verisinin yüksek olduğu yerlerde daha fazla süt içecek..." — evet bu, bu en karışık olan. Neredeyse anlamsız, kendini tekrar ediyor, saçmalıyor.

Bu soruların tutarsızlığı ve anlamsızlığı hakkında bi kaç bi şey söylemek gerekirse: modelimiz küçük oldugu için saçmalıyor olabilir ,modelimiz bi tık daha büyük olsaydı daha anlamlı ve tutartlı sonuclar görebilirdik.




*   yapabileceğimiz birinci değişiklik daha büyük bir model bulmak olabilir.
*   ya da bilgi tabanını zenginleştirebilriiz.Yani her konuya daha fazla bilgi ve anlam yükleyebiliriz . Çünkü sorunumuz modelin konuşması değil,elindeki bilginiin kısa olmasından dolayı olabilir.


Bu iki çözüm yolunu da denemek istiyorum.Önce herhangi birisiyle başlayalım:



In [8]:
# Zenginleştirilmiş bilgi tabanı — daha detaylı metinler
bilgi_tabani = [
    "Buzağıların yem yememesinin yaygın sebepleri şunlardır: geçiş dönemi stresi "
    "(sütten yeme geçiş), sindirim sistemi bozuklukları, diş çıkarma dönemi ağrısı, "
    "soğuk veya kirli su, ani yem değişikliği ve bağırsak parazitleri. Çözüm olarak "
    "yem değişikliği kademeli yapılmalı, temiz su sağlanmalı ve ortam sıcak "
    "tutulmalıdır. İştahsızlık iki günü geçerse veteriner hekime başvurulmalıdır.",

    "Süt ineklerinde süt veriminin düşmesinin başlıca nedenleri: yetersiz veya "
    "kalitesiz yem, su tüketiminin azalması, sıcaklık stresi, meme iltihabı "
    "(mastitis), laktasyon döneminin ilerlemesi ve gebelik. İlk kontrol edilmesi "
    "gerekenler yem kalitesi ve miktarı ile temiz suya erişimdir. Verim ani "
    "düştüyse mastitis açısından meme kontrol edilmeli; süt kıvamı ve rengi "
    "incelenmelidir. Sorun devam ederse veteriner hekime danışılmalıdır.",

    "İneklerde topallık nedenleri: tırnak hastalıkları (çürük, çatlak), sert veya "
    "sürekli ıslak zemin, eklem iltihabı, yaralanma ve mineral eksikliği. Topallayan "
    "hayvan diğerlerinden ayrılabilir, yem tüketimi ve süt verimi düşebilir. Erken "
    "fark edilirse tırnak bakımı ve zemin düzenlemesiyle tedavi başarısı yüksektir. "
    "İlerlemiş durumlarda veteriner müdahalesi gerekir.",

    "Gebe ineklerin doğuma yakın döneminde (kuru dönem) beslenmesi kritiktir. Enerji "
    "ve protein ihtiyacı artar, ancak aşırı besleme doğum güçlüğüne ve metabolik "
    "hastalıklara yol açabilir. Doğumdan yaklaşık 3 hafta önce geçiş rasyonuna "
    "başlanmalı, mineral ve vitamin desteği verilmelidir. Doğum yaklaştığında hayvan "
    "temiz ve rahat bir bölmede gözlem altında tutulmalıdır.",

    "Hayvanlarda aşı takvimine uyulması hastalık önlemenin temelidir. Şap, brusella, "
    "şarbon ve yanıkara gibi hastalıklara karşı düzenli aşılama yapılmalıdır. Aşı "
    "zamanları hayvanın yaşına ve bölgedeki risklere göre veteriner hekim tarafından "
    "belirlenir. Aşı sonrası hayvan birkaç gün gözlem altında tutulmalı, olası yan "
    "etkiler için hazırlıklı olunmalıdır. Aşı kayıtları düzenli tutulmalıdır.",

    "Süt sağımı hijyeni mastitis riskini azaltmanın en önemli yoludur. Sağımdan önce "
    "meme temizlenmeli ve kurulanmalı, sağım ekipmanları her kullanımdan sonra "
    "dezenfekte edilmelidir. İlk sütte pıhtı, kan veya renk değişikliği görülürse "
    "mastitis şüphesiyle veteriner hekime danışılmalıdır. Sağım sonrası memelerin "
    "temiz ortamda tutulması enfeksiyon riskini düşürür."
]

# Yeni bilgi tabanını tekrar embedding'e çevir (önemli!)
bilgi_embed = embed_model.encode(bilgi_tabani, convert_to_tensor=True)

print("Zenginleştirilmiş bilgi tabanı hazır:", len(bilgi_tabani), "metin")
print("Embedding'ler yeniden hesaplandı.")

Zenginleştirilmiş bilgi tabanı hazır: 6 metin
Embedding'ler yeniden hesaplandı.


Evet dediğimiz gibi bilgi tabanını zenginleştirmek istedim ve bu yolu seçtim. Bi bakalım değişecek mi asistanımızın bize verdiği cevaplar.Az önce cenginleştirlmemiş bilgi tabanına sordugum soruların aynısını tekrar sorucam ve karşılaştırıcaz beraber ,belki tbalo olarak bile ekleyebilriz altına.


In [9]:
sorular = [
    "ineğimin sütü azaldı ne yapmalıyım?",
    "hayvanıma ne zaman aşı yaptırmalıyım?",
    "ineğim topallıyor sebebi ne olabilir?"
]

for s in sorular:
    print("SORU:", s)
    print("ASİSTAN:", veteriner_asistan(s))
    print("-" * 60)

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


SORU: ineğimin sütü azaldı ne yapmalıyım?


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ASİSTAN: İpekçe, inek sütünün azaldığı durumda, ilk kontrol etmeniz gereken yem kalitesi ve miktarı olabilir. Bu durumda, ineklerin yemeklerini daha fazla tüketmek veya daha az yemek almak yerine, daha iyi ve daha sağlıklı yemek seçmeyi deneyebilirsiniz. Ayrıca, ineklerin su tüketimi azaldığında, ineklerin içeceklerinin kaliteli olması önemlidir. 

Bu durumda, ineklerin su tüketimi azaldığında, ineklerin içeceklerinin kaliteli olması önemlidir
------------------------------------------------------------
SORU: hayvanıma ne zaman aşı yaptırmalıyım?


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ASİSTAN: Hayvanınızın aşı takvimini düzenleyerek, onu sağlıklı hale getirmek ve riskleri önlemek için uygun şekilde aşı yapmanız gerekmektedir. Bu durumda, hayvanın yaşını, bölgenizin risk düzeyini ve veterinere başvurmanız gereken diğer faktörlerden bahsedildiğini unutmayın. Eğer bu bilgileri tam olarak kastediyorsanız, aşı yapmanızı öneririm. Ancak, her hayvan için özel bir aşı takvimini belirleyebilir veya belirsizlik varsa, veterinariane başvurmanızı tavsiye ederim.
------------------------------------------------------------
SORU: ineğim topallıyor sebebi ne olabilir?
ASİSTAN: İneklerin topallığını önlemek için en iyi yöntem, tırnak hastalıklarının kontrolü ve düzenli olarak düzenlenmiş ıslak zemin sağlanmasıdır. Bu, ikinci aşamada veterinere başvurmanız önerilir. Eğer ikinci aşamada bile problem çözünmediyseniz, daha uzun vadeli bir tedavide bulunmakta olabilirsiniz. Ancak, herhangi bir hastalığın olduğunu belirtmek için en doğru yol veterinere başvurmanızdır.
-------------------

gördüğümüz üzere bilgi tabanını değiştirmek gözle görülür şekilde anlam karışıklılıgını azaltttı ve az öncekine nispeten mantıklı cevaplar vermeye başladı.Ama bakıldıgında yine ara ara saçmalama ve garip kelime kullanımları da görülmekte .Ama nispeten az öncekine bakılırsa gayet gayet iyi durumda.

Şimdi de büyük model kullanmak denemek istiyorum "`out of memory hatası`" alsam bile denemiş olucam. Belki de daha faydalı anlamlı ve karışık olmayan güzel bi modeli denemiş ve başarmış oluruz.

In [10]:
# Daha büyük model dene (3B) — daha akıcı cevaplar için
llm = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-3B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto"
)

print("3B model yüklendi. Şimdi aynı soruları tekrar test et.")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

3B model yüklendi. Şimdi aynı soruları tekrar test et.


In [11]:
sorular = [
    "ineğimin sütü azaldı ne yapmalıyım?",
    "hayvanıma ne zaman aşı yaptırmalıyım?",
    "ineğim topallıyor sebebi ne olabilir?"
]

for s in sorular:
    print("SORU:", s)
    print("ASİSTAN:", veteriner_asistan(s))
    print("-" * 60)

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


SORU: ineğimin sütü azaldı ne yapmalıyım?


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ASİSTAN: İnekinizin sütü azaldığına dair ilk ve en önemli adım yem ve su için uygun olanları kontrol etmek. Yem kalitesini ve miktarını kontrol etmeniz gerekecek. Ayrıca, ineğin kafasının ve ağızının temiz olması önemlidir. 

İnekinizin sütü azaldığında, mastitis (meme iltihabı) olabileceğini düşünüyorum. Bu durumda, ineğin memesini kontrol etmeniz gerekecek. Memeyi kontrol etmeden önce, ineğin süt kıvamı ve rengini incelemeniz öneriliyor.
------------------------------------------------------------
SORU: hayvanıma ne zaman aşı yaptırmalıyım?


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ASİSTAN: Çiftçi adlı kişi, hayvanının aşı zamanını hayvanın yaşına ve bölgedeki risklere göre belirleyen bir veteriner hekiminden öğrenmelisiniz. Bu bilgi, hayvanın aşı taramalarını düzenleyen hekim tarafından verilecektir. Ancak genel olarak, genellikle 2-3 ay içinde ilk aşı yapılır ve ardından her 1-2 yıl arasında tekrar aşılar yapılır. Her aşı taramasında, hekim, hayvanın sağlıklı olduğunu kontrol eder ve aşı sonrası hayvanı birkaç gün gözlem altında tutmanızı önerir. Aşı kayıtları düzenli tut
------------------------------------------------------------
SORU: ineğim topallıyor sebebi ne olabilir?
ASİSTAN: İneklerin topallığı için birkaç neden olabilir. İlk olarak, tırnaklarında çatlak veya çürük olabilir. Ayrıca, zemin sert veya sürekli ıslak olabilir. Eklem iltihabı da bir sebep olabilir. Yaralanma ve minerallerin eksikliği de topallığın nedeni olabilir. Eğer ineğin bu nedenlerden biri olduğunu düşünüyorsanız, tırnaklarını temizleyip kontrol etmenizi öneririm. Eğer iyileşmeden sonr

3b modele geçince gelen değişiklik gerçekten mutlu edici.Gördüğünüz gibi anlam karışıklığı yok sorumlukluklarının farkında (veteriner hekime danışın ,takviminizi ayarlayın ).Anlam kaırıklıgı ve gereksiz kelime kullanımı nerdeyse yok denecek akdar az.Korktuğum olmadı "out of memory" hatası alır mıyım diye korktum .Ama cok şükür korktugum gibi olmadı gayet güzel bi sonuc almıs olduk.

`Yaptığımız iyilrştirmeler şunlardı:`



1.)Bilgi tabanını zenginleştirdik → cevaplar doğru konuya oturdu.

2.)3B modele geçtik → cevaplar akıcı, tekrarsız, profesyonel oldu.


Bilgi tabanını zenginleştirdik → cevaplar doğru konuya oturdu.
3B modele geçtik → cevaplar akıcı, tekrarsız, profesyonel oldu.

##  Kapanış — RAG Tabanlı Veteriner Asistanı

Bu notebook'ta, çiftçinin doğal dille soru sorup cevap alabileceği bir yapay zeka
asistanı kurduk. Yöntem olarak RAG (Retrieval-Augmented Generation) kullandık.

**İzlenen adımlar**
1. Veteriner bilgi tabanı oluşturma (hastalık, beslenme, aşı, sağım konuları)
2. Metinleri embedding'e (anlam vektörü) çevirme — sentence-transformers
3. Soruyu bilgi tabanıyla kosinüs benzerliğiyle eşleştirip en uygun bilgiyi bulma
4. Bir dil modeline (Qwen2.5) bu bilgiyi verip sade bir cevap ürettirme
5. Bilgi tabanını zenginleştirme ve modeli 1.5B'den 3B'ye büyüterek kaliteyi artırma

**Bulgular**
- RAG mimarisi baştan sona çalıştı: soru → doğru bilgi bulma → cevap üretme.
- Cevap kalitesi iki şeye bağlı çıktı: bilgi tabanının zenginliği ve model boyutu.
  Bilgi tabanını detaylandırmak cevapları doğru konuya oturttu; 3B modele geçmek
  akıcılığı ve tutarlılığı belirgin artırdı.
- Asistan, sistem talimatı sayesinde teşhis koymuyor ve her durumda veteriner
  hekime yönlendiriyor — hayvan sağlığı için kritik bir sorumluluk önlemi.

**Projedeki karşılığı:** Çiftçinin uygulamadan soru sorup bilgi alabileceği
"konuşan asistan" özelliğinin çalışan prototipi. Gerçek projede bilgi tabanı
gerçek veteriner dökümanlarıyla genişletilerek güçlendirilebilir.

In [13]:
# Bilgi tabanı 24 konu · PY
# =====================================================================
# VETERİNER ASİSTANI — BİLGİ TABANI (24 konu)
# ---------------------------------------------------------------------
# NOT: Aşağıdaki metinler hayvancılık/veterinerlik alanında genel-geçer,
# temel bilgilerdir. Kesin tıbbi teşhis kaynağı DEĞİLDİR; asistan her
# durumda kullanıcıyı veteriner hekime yönlendirir. İleride gerçek
# veteriner dökümanlarıyla genişletilebilir/değiştirilebilir.
# =====================================================================

bilgi_tabani = [
    # 1 — Buzağı iştahsızlığı
    "Buzağıların yem yememesinin yaygın sebepleri geçiş dönemi stresi, sindirim "
    "bozuklukları, diş çıkarma ağrısı, soğuk veya kirli su, ani yem değişikliği ve "
    "bağırsak parazitleridir. Yem değişikliği kademeli yapılmalı, temiz su sağlanmalı "
    "ve ortam sıcak tutulmalıdır. İştahsızlık iki günü geçerse veteriner hekime "
    "başvurulmalıdır.",

    # 2 — Süt verimi düşüşü
    "Süt veriminin düşmesinin başlıca nedenleri yetersiz veya kalitesiz yem, su "
    "tüketiminin azalması, sıcaklık stresi, meme iltihabı (mastitis), laktasyon "
    "döneminin ilerlemesi ve gebeliktir. İlk kontrol edilmesi gerekenler yem kalitesi, "
    "yem miktarı ve temiz suya erişimdir. Ani düşüşte meme mastitis açısından kontrol "
    "edilmeli, sorun sürerse veteriner hekime danışılmalıdır.",

    # 3 — Topallık
    "İneklerde topallık genellikle tırnak hastalıkları (çürük, çatlak), sert veya "
    "sürekli ıslak zemin, eklem iltihabı, yaralanma ve mineral eksikliğinden "
    "kaynaklanır. Topallayan hayvanın yem tüketimi ve süt verimi düşebilir. Erken fark "
    "edilirse tırnak bakımı ve zemin düzenlemesiyle tedavi başarısı yüksektir; "
    "ilerlemiş durumlarda veteriner müdahalesi gerekir.",

    # 4 — Gebelik ve kuru dönem beslenmesi
    "Gebe ineklerin doğuma yakın kuru döneminde beslenmesi kritiktir. Enerji ve protein "
    "ihtiyacı artar; ancak aşırı besleme doğum güçlüğüne ve metabolik hastalıklara yol "
    "açabilir. Doğumdan yaklaşık üç hafta önce geçiş rasyonuna başlanmalı, mineral ve "
    "vitamin desteği verilmelidir. Doğum yaklaştığında hayvan temiz ve rahat bir "
    "bölmede gözlem altında tutulmalıdır.",

    # 5 — Aşılama
    "Aşı takvimine uyulması hastalıkların önlenmesinin temelidir. Şap, brusella, şarbon "
    "ve yanıkara gibi hastalıklara karşı düzenli aşılama yapılmalıdır. Aşı zamanları "
    "hayvanın yaşına ve bölgedeki risklere göre veteriner hekim tarafından belirlenir. "
    "Aşı sonrası hayvan birkaç gün gözlem altında tutulmalı ve aşı kayıtları düzenli "
    "olarak tutulmalıdır.",

    # 6 — Sağım hijyeni ve mastitis
    "Süt sağımı hijyeni mastitis (meme iltihabı) riskini azaltmanın en önemli yoludur. "
    "Sağımdan önce meme temizlenip kurulanmalı, sağım ekipmanları her kullanımdan sonra "
    "dezenfekte edilmelidir. İlk sütte pıhtı, kan veya renk değişikliği görülürse "
    "mastitis şüphesiyle veteriner hekime danışılmalıdır. Sağım sonrası memelerin temiz "
    "ortamda tutulması enfeksiyon riskini düşürür.",

    # 7 — İshal (diyare)
    "İshal, özellikle buzağılarda tehlikelidir ve hızlı sıvı kaybına yol açar. Nedenleri "
    "arasında bakteri, virüs veya parazit enfeksiyonları, ani yem değişikliği, kirli su "
    "ve hijyen eksikliği bulunur. Hayvana bol temiz su ve gerekirse elektrolit "
    "verilmelidir. İshal bir günden uzun sürerse, kanlıysa veya hayvan halsizse acilen "
    "veteriner hekime başvurulmalıdır.",

    # 8 — Su ihtiyacı
    "Yeterli ve temiz su, hayvan sağlığı ve süt verimi için kritiktir. Bir süt ineği "
    "günde yaklaşık 60-100 litre su içebilir; su kısıtlandığında süt verimi hızla "
    "düşer. Su kaynağı temiz, kolay erişilebilir ve sıcak havalarda serin tutulmalıdır. "
    "Kirli veya yetersiz su iştahsızlık ve hastalıklara zemin hazırlar.",

    # 9 — Parazitler
    "İç ve dış parazitler hayvanlarda zayıflama, kıl dökülmesi, kaşıntı, kansızlık ve "
    "verim düşüklüğüne yol açar. İç parazitler için ilaçlama, dış parazitler için uygun "
    "uygulamalar düzenli olarak yapılmalıdır. Parazit kontrol programı, mevsime ve bölge "
    "koşullarına göre veteriner hekim önerisiyle planlanmalıdır.",

    # 10 — Doğum (buzağılama)
    "Doğum sırasında hayvan sakin, temiz ve gözlem altında tutulmalıdır. Normal doğum "
    "genellikle birkaç saat içinde tamamlanır. Doğum uzarsa, buzağının duruşu ters ise "
    "veya hayvan aşırı zorlanıyorsa vakit kaybetmeden veteriner hekim çağrılmalıdır. "
    "Doğum sonrası hem ana hem yavru yakından izlenmelidir.",

    # 11 — Kolostrum (ağız sütü)
    "Yeni doğan buzağının ilk saatlerde ağız sütü (kolostrum) alması hayati önemdedir. "
    "Kolostrum buzağıya bağışıklık kazandırır ve hastalıklara karşı korur. İlk iki saat "
    "içinde, doğum ağırlığının yaklaşık yüzde onu kadar kolostrum verilmelidir. Geciken "
    "veya yetersiz kolostrum, buzağının hastalanma ve ölüm riskini belirgin artırır.",

    # 12 — Sıcaklık stresi
    "Sıcaklık stresi, özellikle yaz aylarında süt ineklerinde verim düşüşüne, "
    "iştahsızlığa ve solunum hızlanmasına yol açar. Hayvanlara gölgelik, iyi "
    "havalandırma ve bol serin su sağlanmalıdır. Sıcak saatlerde ağır yem yerine daha "
    "hafif ve sindirilebilir beslenme tercih edilmelidir. Aşırı sıcak hayvan sağlığı "
    "için ciddi bir risktir.",

    # 13 — Rasyon ve dengeli besleme
    "Hayvanların sağlıklı olması ve verimli çalışması için dengeli bir rasyon şarttır. "
    "Rasyon; enerji, protein, lif, mineral ve vitaminleri hayvanın ihtiyacına göre "
    "içermelidir. Kaba yem (ot, silaj) ve kesif yem (tahıl karması) dengesi önemlidir. "
    "Dengesiz besleme, verim düşüklüğüne ve sindirim sorunlarına yol açar.",

    # 14 — Şişkinlik (timpani)
    "Şişkinlik (timpani), işkembede aşırı gaz birikmesiyle oluşan, hızlı gelişebilen "
    "ciddi bir durumdur. Genellikle aşırı taze/yaş yonca gibi baklagillerin fazla "
    "tüketilmesiyle görülür. Hayvanın sol böğrü belirgin şişer, huzursuzluk ve solunum "
    "güçlüğü olur. Şişkinlik acil bir durumdur; derhal veteriner hekime başvurulmalıdır.",

    # 15 — Ayak ve tırnak bakımı
    "Düzenli tırnak bakımı topallığı ve ayak hastalıklarını önlemenin temelidir. "
    "Tırnaklar aşırı uzadığında hayvanın duruşu bozulur ve yürüme güçleşir. Ahır zemini "
    "kuru, temiz ve kaymayı önleyecek şekilde olmalıdır. Yılda birkaç kez tırnak kesimi "
    "ve kontrolü önerilir; belirgin aksama varsa veteriner hekime danışılmalıdır.",

    # 16 — Kızgınlık (östrus) takibi
    "Kızgınlık (östrus) belirtilerinin doğru takibi, başarılı tohumlama için gereklidir. "
    "Belirtiler arasında huzursuzluk, diğer hayvanlara atlama veya atlanmaya izin verme, "
    "iştah değişikliği ve akıntı bulunur. Kızgınlık genellikle belirli aralıklarla "
    "tekrarlar. Doğru zamanda tohumlama için kızgınlık günü kayıt altına alınmalıdır.",

    # 17 — Buzağı barınağı ve hijyen
    "Buzağılar bağışıklıkları zayıf olduğu için temiz, kuru ve rüzgârdan korunaklı bir "
    "barınakta tutulmalıdır. Islak ve kirli zemin, ishal ve solunum hastalıklarına "
    "davetiye çıkarır. Barınak düzenli temizlenmeli, altlık kuru tutulmalı ve yeterli "
    "temiz hava sağlanmalıdır. Hasta buzağılar sağlıklı olanlardan ayrılmalıdır.",

    # 18 — Solunum yolu hastalıkları
    "Solunum yolu hastalıkları, özellikle genç hayvanlarda öksürük, burun akıntısı, "
    "hızlı solunum ve ateşle kendini gösterir. Nedenleri arasında soğuk, nemli ve kötü "
    "havalandırılan barınaklar, ani sıcaklık değişimleri ve enfeksiyonlar bulunur. Erken "
    "fark edilirse tedavi başarılıdır; belirtiler görülürse veteriner hekime "
    "başvurulmalıdır.",

    # 19 — Mineral ve vitamin eksikliği
    "Mineral ve vitamin eksiklikleri; iştahsızlık, zayıflama, tüy/kıl bozuklukları, "
    "üreme sorunları ve verim düşüklüğüne yol açabilir. Özellikle kalsiyum, fosfor, "
    "selenyum ve A, D, E vitaminleri önemlidir. Dengeli rasyon ve gerektiğinde mineral "
    "takviyesiyle önlenebilir. Takviye programı veteriner hekim önerisiyle "
    "belirlenmelidir.",

    # 20 — Süt humması (doğum felci)
    "Süt humması (doğum felci), genellikle doğumdan hemen sonra kandaki kalsiyumun ani "
    "düşmesiyle görülür. Hayvan halsizleşir, ayağa kalkamaz ve titreme görülebilir. "
    "Özellikle yüksek verimli ve yaşlı ineklerde risk daha yüksektir. Bu acil bir "
    "durumdur; derhal veteriner hekime başvurulmalıdır.",

    # 21 — Yem değişikliği ve sindirim sağlığı
    "Yemdeki ani değişiklikler işkembe dengesini bozarak sindirim sorunlarına, "
    "iştahsızlığa ve verim düşüklüğüne yol açar. Yeni bir yeme geçiş birkaç gün içinde "
    "kademeli yapılmalıdır. İşkembe sağlığı için yeterli kaba yem (lif) verilmesi "
    "önemlidir. Ani ve aşırı kesif yem, asidoz gibi sorunlara neden olabilir.",

    # 22 — Genel sağlık gözlemi
    "Hayvanların günlük gözlemi, sorunların erken fark edilmesini sağlar. İştah, "
    "hareketlilik, dışkı kıvamı, süt verimi ve davranıştaki değişimler önemli "
    "göstergelerdir. Kulakların düşük olması, sürüden ayrı durma, iştahsızlık veya "
    "verim düşüşü dikkat edilmesi gereken işaretlerdir. Şüpheli durumlarda veteriner "
    "hekime danışılmalıdır.",

    # 23 — Su ve yem kabı temizliği
    "Yem ve su kaplarının düzenli temizliği, hastalıkların yayılmasını önler. Kirli "
    "kaplarda bakteri ve küf üreyebilir; bu da iştahsızlık ve sindirim sorunlarına yol "
    "açar. Su kapları her gün, yemlikler düzenli aralıklarla temizlenmelidir. Küflenmiş "
    "veya bozulmuş yem kesinlikle hayvana verilmemelidir.",

    # 24 — Yeni hayvan katılımı ve karantina
    "Sürüye yeni katılan hayvanlar, hastalık taşıma riskine karşı bir süre karantinada "
    "(ayrı) tutulmalıdır. Bu sürede hayvan gözlemlenmeli, gerekli aşı ve parazit "
    "kontrolleri yapılmalıdır. Karantina, bulaşıcı hastalıkların tüm sürüye yayılmasını "
    "önleyen önemli bir koruyucu tedbirdir. Şüpheli belirtilerde veteriner hekime "
    "danışılmalıdır."
]

# Bilgi tabanını embedding'e çevir (bilgi tabanı her değiştiğinde ZORUNLU)
bilgi_embed = embed_model.encode(bilgi_tabani, convert_to_tensor=True)

print("Bilgi tabanı hazır:", len(bilgi_tabani), "konu")
print("Embedding'ler hesaplandı.")

Bilgi tabanı hazır: 24 konu
Embedding'ler hesaplandı.


Ben gerekli çalışmalarımı yaptıktan sonra boş vaktimi değerlendirip gün içeriisinde asistanımı daha da geliştirmek istedim ve öncelikle bilgi_tabanı metnimi daha da arttırmka istedim yapay zekaya yazdırdıgım 24 farklı genel veterinerlik bilgilerini ekleyip embedding uygulamaya çalıstım

In [17]:
def veteriner_asistan(soru, esik=0.20):
    soru_embed = embed_model.encode(soru, convert_to_tensor=True)
    benzerlikler = util.cos_sim(soru_embed, bilgi_embed)[0]

    if benzerlikler.max().item() < esik:
        return ("Bu konuda bilgi tabanımda yeterli bilgi yok. Lütfen hayvan "
                "sağlığıyla ilgili bir soru sorun veya veteriner hekime danışın.")

    en_iyi_index = benzerlikler.argsort(descending=True)[:2]
    secili_bilgiler = "\n\n".join([bilgi_tabani[i] for i in en_iyi_index])

    mesaj = [
        {"role": "system", "content":
         "Sen bir veteriner asistanısın. SADECE sana verilen bilgilere dayanarak "
         "cevap ver. Verilen bilgilerden soruyla EN ALAKALI olanı kullan, alakasız "
         "olanı görmezden gel. Bilgide olmayan şeyi uydurma. Teşhis koyma ve sonunda "
         "mutlaka veteriner hekime danışılmasını öner."},
        {"role": "user", "content":
         f"Bilgiler:\n{secili_bilgiler}\n\nÇiftçinin sorusu: {soru}"}
    ]

    cevap = llm(mesaj, max_new_tokens=200, do_sample=False)
    return cevap[0]["generated_text"][-1]["content"]


# Test
for s in ["buzağımda ishal var ne yapmalıyım?", "ineğimin sütü azaldı", "telefonum bozuldu"]:
    print("SORU:", s)
    print("ASİSTAN:", veteriner_asistan(s))
    print("=" * 60)

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


SORU: buzağımda ishal var ne yapmalıyım?


[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ASİSTAN: Buzağımda ishal varsa, bu bir hijyeni mastitis riskini artırabilir. Ancak, verilen bilgilerde buzağımda ishal varsa ne yapılacağına dair spesifik bir öneride bulunulmamış. Hijyeni mastitis riskini azaltmak için sağımın ve memelerin uzun vadeli ve düzenli hijyen performansı önemlidir. Ancak, buzağımda ishal varsa, bu da bir hastalığın başlamasına neden olabileceğini gösteriyor. Bu nedenle, bu durumu görür görmez, veteriner hekimine başvurmanızı öneririm.
SORU: ineğimin sütü azaldı
ASİSTAN: Verdiğiniz bilgiler ve sorunuzna dayanılarak, ineğinizin süt veriminin düşmesinin en alakalı nedeni yetersiz veya kalitesiz yem gibi faktörlerdir. İlk kontrol edilmesi gerekenler arasında yem kalitesi ve yem miktarı yer almaktadır. Ancak, bu tür bir durumun anında iyileşmesi zor olabilir ve hemiyen bir bakış açısı gerektirebilir. Bu nedenle, anında süt veriminin düşmesine neden olan bu faktörleri kontrol etmek için ilk olarak yem kalitesini ve miktarını kontrol etmenizi öneririm. Ancak, bu da

Bu güzel kodumuzun amacı esik kontrolü yapıyoruz.Soruyu tüm metinlerle karşılaştırılıp en yüksek benzerliğe sahip olana bakıyoruz.Eger bi metinle bile alakası yoksa bu konu hakkında bilgim yok diyor (yani cevap uyudurup sallamıyor)


`argsort(descending=True)[:2]` ile en yakın 2 metni alıyoruz (eskiden 1 taneydi). Böylece asistan daha kapsamlı cevap verebiliyor; mesela ishal sorusuna hem ishal hem su hem barınak bilgisini birleştirebiliyor.



In [19]:
test_sorulari = [
    "buzağımda ishal var ne yapmalıyım?",
    "ineğimin sütü azaldı",
    "bugün hava nasıl olacak?",
    "telefonum bozuldu"
]

for s in test_sorulari:
    e = embed_model.encode(s, convert_to_tensor=True)
    skor = util.cos_sim(e, bilgi_embed)[0].max().item()
    print(f"{s:40} → en yüksek benzerlik: {round(skor, 3)}")

buzağımda ishal var ne yapmalıyım?       → en yüksek benzerlik: 0.263
ineğimin sütü azaldı                     → en yüksek benzerlik: 0.752
bugün hava nasıl olacak?                 → en yüksek benzerlik: 0.316
telefonum bozuldu                        → en yüksek benzerlik: 0.115


Şimdi RAG testine baktığımda, eşik değerimi değiştirerek (0.20, 0.30, 0.35)
sonuçlarımı analiz ettim. `argsort(descending=True)[:2]` kodum ile de sorduğum
soruyu tüm metinlerle karşılaştırıp en yakın 2 metne göre cevap verme işini
yaptım. Fakat gördüğüm üzere her seferinde ufak tefek hatalar aldım; bazen
"bilgim yok" demesi gerekirken bile yorum yaptı.

İşte tam burada, RAG'ın tek başına %100 çalışmadığını fark ederek LLM eklemeye
karar verdim. LLM, gelen soruyu önce "bu seninle (hayvan sağlığıyla) ilgili mi?"
diye kontrol eder; gerçekten alakasız bir şeyse soruyu direkt reddeder.

LLM'ler çok daha akıllı olduğu için daha iyi sonuç alacağımı umuyorum ve bu
yaklaşımı denemek istiyorum.

In [23]:
def veteriner_asistan(soru):
    # 1. LLM ön kontrol (few-shot ile, çalışan sürüm)
    if not konu_uygun_mu(soru):
        return ("Ben bir veteriner asistanıyım ve yalnızca hayvan sağlığı ile ilgili "
                "sorulara yardımcı olabilirim. Lütfen hayvanlarınızla ilgili bir soru sorun.")

    # 2. RAG: en yakın 2 metin
    soru_embed = embed_model.encode(soru, convert_to_tensor=True)
    benzerlikler = util.cos_sim(soru_embed, bilgi_embed)[0]
    en_iyi_index = benzerlikler.argsort(descending=True)[:2]
    secili_bilgiler = "\n\n".join([bilgi_tabani[i] for i in en_iyi_index])

    # 3. Cevap üret
    mesaj = [
        {"role": "system", "content":
         "Sen bir veteriner asistanısın. SADECE sana verilen bilgilere dayanarak "
         "cevap ver. Soruyla EN ALAKALI bilgiyi kullan. Bilgide olmayanı uydurma. "
         "Teşhis koyma ve sonunda mutlaka veteriner hekime danışılmasını öner."},
        {"role": "user", "content":
         f"Bilgiler:\n{secili_bilgiler}\n\nÇiftçinin sorusu: {soru}"}
    ]
    cevap = llm(mesaj, max_new_tokens=200, do_sample=False)
    return cevap[0]["generated_text"][-1]["content"]


# Tam sistem testi
for s in ["buzağımda ishal var ne yapmalıyım?", "bugün hava nasıl olacak?"]:
    print("SORU:", s)
    print("ASİSTAN:", veteriner_asistan(s))
    print("=" * 60)

[transformers] Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


SORU: buzağımda ishal var ne yapmalıyım?


[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ASİSTAN: Buzağımda ishal varsa, bu bir hastalığın başlangıcını gösteriyor olabilir. Ancak, verilen bilgilerde buzağımda ishal varsa ne yapılacağını belirtmedik. Hijyeni mastitis riskini azaltmak için sağımın ve memelerin uzun vadeli ve temiz bir ortamda tutulması önemlidir. Ancak, buzağımda ishal varsa, bu bir hastalığın başlangıcını gösteriyor olabilir ve bu durumda en doğru adım veteriner hekimine başvurmak olur. Herhangi bir hastalığın ilk vurgusunu göründüğünde, profesyonel sağlık ajansına başvurmak önemlidir.
SORU: bugün hava nasıl olacak?
ASİSTAN: Ben bir veteriner asistanıyım ve yalnızca hayvan sağlığı ile ilgili sorulara yardımcı olabilirim. Lütfen hayvanlarınızla ilgili bir soru sorun.


In [25]:
def veteriner_asistan(soru):
    # 1. LLM ön kontrol (few-shot ile, çalışan sürüm)
    if not konu_uygun_mu(soru):
        return ("Ben bir veteriner asistanıyım ve yalnızca hayvan sağlığı ile ilgili "
                "sorulara yardımcı olabilirim. Lütfen hayvanlarınızla ilgili bir soru sorun.")

    # 2. RAG: en yakın 2 metin
    soru_embed = embed_model.encode(soru, convert_to_tensor=True)
    benzerlikler = util.cos_sim(soru_embed, bilgi_embed)[0]
    en_iyi_index = benzerlikler.argsort(descending=True)[:2]
    secili_bilgiler = "\n\n".join([bilgi_tabani[i] for i in en_iyi_index])

    # 3. Cevap üret
    mesaj = [
        {"role": "system", "content":
         "Sen bir veteriner asistanısın. SADECE sana verilen bilgilere dayanarak "
         "cevap ver. Soruyla EN ALAKALI bilgiyi kullan. Bilgide olmayanı uydurma. "
         "Teşhis koyma ve sonunda mutlaka veteriner hekime danışılmasını öner."},
        {"role": "user", "content":
         f"Bilgiler:\n{secili_bilgiler}\n\nÇiftçinin sorusu: {soru}"}
    ]
    cevap = llm(mesaj, max_new_tokens=200, do_sample=False)
    return cevap[0]["generated_text"][-1]["content"]


# Tam sistem testi
for s in ["buzağımda ishal var ne yapmalıyım?", "bugün hava nasıl olacak?"]:
    print("SORU:", s)
    print("ASİSTAN:", veteriner_asistan(s))
    print("=" * 60)

[transformers] Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


SORU: buzağımda ishal var ne yapmalıyım?


[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ASİSTAN: Buzağımda ishal varsa, bu bir hastalığın başlangıcını gösteriyor olabilir. Ancak, verilen bilgilerde buzağımda ishal varsa ne yapılacağını belirtmedik. Hijyeni mastitis riskini azaltmak için sağımın ve memelerin uzun vadeli ve temiz bir ortamda tutulması önemlidir. Ancak, buzağımda ishal varsa, bu bir hastalığın başlangıcını gösteriyor olabilir ve bu durumda en doğru adım veteriner hekimine başvurmak olur. Herhangi bir hastalığın ilk vurgusunu göründüğünde, profesyonel sağlık ajansına başvurmak önemlidir.
SORU: bugün hava nasıl olacak?
ASİSTAN: Ben bir veteriner asistanıyım ve yalnızca hayvan sağlığı ile ilgili sorulara yardımcı olabilirim. Lütfen hayvanlarınızla ilgili bir soru sorun.


In [24]:
test_sorulari = [
    "buzağımda ishal var ne yapmalıyım?",
    "ineğimin sütü azaldı sebebi ne olabilir?",
    "hayvanıma ne zaman aşı yaptırmalıyım?",
    "ineğim topallıyor",
    "gebe ineğimi nasıl beslemeliyim?",
    "bugün hava nasıl olacak?",
    "telefonum bozuldu ne yapmalıyım?",
    "en yakın market nerede?",
    "buzağıya ilk saatlerde ne vermeliyim?",
    "futbol maçı kaçta başlıyor?"
]

for s in test_sorulari:
    print("SORU:", s)
    print("ASİSTAN:", veteriner_asistan(s))
    print("=" * 60)

[transformers] Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


SORU: buzağımda ishal var ne yapmalıyım?


[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ASİSTAN: Buzağımda ishal varsa, bu bir hastalığın başlangıcını gösteriyor olabilir. Ancak, verilen bilgilerde buzağımda ishal varsa ne yapılacağını belirtmedik. Hijyeni mastitis riskini azaltmak için sağımın ve memelerin uzun vadeli ve temiz bir ortamda tutulması önemlidir. Ancak, buzağımda ishal varsa, bu bir hastalığın başlangıcını gösteriyor olabilir ve bu durumda en doğru adım veteriner hekimine başvurmak olur. Herhangi bir hastalığın ilk vurgusunu göründüğünde, profesyonel sağlık ajansına başvurmak önemlidir.
SORU: ineğimin sütü azaldı sebebi ne olabilir?


[transformers] Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

ASİSTAN: Ben bir veteriner asistanıyım ve yalnızca hayvan sağlığı ile ilgili sorulara yardımcı olabilirim. Lütfen hayvanlarınızla ilgili bir soru sorun.
SORU: en yakın market nerede?


[transformers] Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ASİSTAN: Ben bir veteriner asistanıyım ve yalnızca hayvan sağlığı ile ilgili sorulara yardımcı olabilirim. Lütfen hayvanlarınızla ilgili bir soru sorun.
SORU: buzağıya ilk saatlerde ne vermeliyim?


[transformers] Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ASİSTAN: Buzağa ilk saatlerde kolostrum vermeliyiz. Kolostrum, buzağının bağışıklık sistemini güçlendirmeye yardımcı olur ve hastalıklarla mücadele etmesi için gereklidir. Bilgilerde belirtildiği gibi, ilk iki saat içinde doğum ağırlığının yaklaşık yüzde onu kadar kolostrum verilmelidir. Bu, buzağın sağlıklı bir yaşam süresince daha fazla sağlığı kazanmasına yardımcı olur. Ancak, buzağın sağlığı hakkında daha fazla bilgi almak için profesyonel bir veteriner ile görüşmeniz önemlidir.
SORU: futbol maçı kaçta başlıyor?
ASİSTAN: Ben bir veteriner asistanıyım ve yalnızca hayvan sağlığı ile ilgili sorulara yardımcı olabilirim. Lütfen hayvanlarınızla ilgili bir soru sorun.


Burada biraz neler yaptığımı sizlere anlatmaya çalışıcam .Bildiğiniz üzere rag la cok iyi sonuclar alamayı llm e gececğimizi söylemiştik.LLM de ilk başta dogru sonuclar alamasak da gerkeli düzenlemeler yapıldıgında gayet iyi(10 üzerinden 9 u dogru) sonuclarımızı almıs olduk bu da gayet iyi gibi görünüyor ilk başta.

Sonrasında dediğimiz gibi test amaçlı 10 adet soru sorup denedim ve gayet iyi ve tutarlı bi sonuc cıkardı.İstediğime ulaştıgım için mutluyum.


Bugun ilk basta 6 metinle başladıgımız asistan colabımızı geliştirerek 24 kaynağa çıkardık ve sonrasında çoklu kaynak(en yakın  2-3 metine dayanarak) güzel cevaplar vermesini sağladık.Ama bazen yanlıs cevaplar verince de rag ı bırakıp llm i denemek istedim.Ve güzel sonuclarla karsılastık ve basardık :::::::)))))))))))))))

In [26]:
import joblib

# Güncel bilgi tabanı + embedding'lerini kaydet
asistan_verisi = {
    "bilgi_tabani": bilgi_tabani,
    "embeddingler": bilgi_embed.cpu().numpy()
}

joblib.dump(asistan_verisi, "asistan_bilgi_tabani.pkl")
print("Kaydedildi: asistan_bilgi_tabani.pkl")
print("Konu sayısı:", len(bilgi_tabani))

Kaydedildi: asistan_bilgi_tabani.pkl
Konu sayısı: 24


##  Kapanış — Gelişmiş RAG + LLM Veteriner Asistanı

Bu notebook'ta, çiftçinin doğal dille soru sorabileceği akıllı bir veteriner
asistanı kurduk ve adım adım geliştirdik.

**Yapılanlar**
- Bilgi tabanı 6 konudan 24 konuya genişletildi (buzağı bakımı, mastitis, ishal,
  şişkinlik, süt humması, aşı, beslenme vb.).
- Çoklu kaynak eklendi: soruya en yakın 2 metin birlikte kullanılıyor.
- Alakasız soruları elemek için önce embedding eşiği denendi; ancak skorlar
  yanıltıcı çıktığı için (ör. "hava nasıl" sorusu gerçek bir sorudan yüksek skor
  aldı) bu yöntem tek başına yeterli olmadı.
- Çözüm olarak LLM ön kontrolü eklendi: model önce sorunun hayvan sağlığıyla ilgili
  olup olmadığına karar veriyor. İlk denemede model kararsız kaldı; few-shot
  (örnekli) yönlendirmeyle sorun çözüldü.

**Sonuç**
10 soruluk testte (6 gerçek + 4 alakasız) asistan 10'da 9 doğru sonuç verdi: tüm
alakasız sorular reddedildi, gerçek sorular doğru cevaplandı (yalnızca örneklerde
bulunmayan "aşı" sorusu kaçırıldı). Bilgi tabanı ve embedding'ler
`asistan_bilgi_tabani.pkl` olarak kaydedildi.

**Not:** Bilgi tabanı genel bilgilerden oluşur, kesin tıbbi kaynak değildir; asistan
her cevabında veteriner hekime yönlendirir. Gerçek projede bilgi tabanı profesyonel
veteriner dökümanlarıyla genişletilebilir.